<a href="https://colab.research.google.com/github/mnaredla-cloud/GenAI/blob/main/RAG/2)RAG_Passing_Context_By_Reading_File.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [54]:
########Passing context from a file################

In [55]:
pip install langchain_community

In [56]:
import os
from langchain_community.document_loaders import TextLoader #importing TextLoader
from sentence_transformers import SentenceTransformer #importing SentenceTransformer
from langchain_community.vectorstores import Chroma #importing Chroma vector_DB
from langchain_community.embeddings import HuggingFaceEmbeddings #embedding models from Hugging Face
from transformers import pipeline #for LLM
from langchain_text_splitters import MarkdownHeaderTextSplitter

In [57]:
#Step_1: Loading a file

from langchain_community.document_loaders import TextLoader
loader = TextLoader('/content/sample_data/tennis_details.md')
text_doc = loader.load()
#print(text_doc[0].page_content)

In [58]:
#Step_2: devide the data into chunks

from langchain_text_splitters import MarkdownHeaderTextSplitter
split_condition = [('##','title')] #split at ##, say it as title
splitter = MarkdownHeaderTextSplitter(split_condition)
doc_splits = splitter.split_text(text_doc[0].page_content)

#get the splits
print(doc_splits)

text_chunks = [split.page_content for split in doc_splits]
#get the chunks
print(text_chunks)

print(len(text_chunks))

[Document(metadata={}, page_content='# Tennis'), Document(metadata={'title': 'Introduction'}, page_content="Tennis is a popular sport played between two players (singles) or two teams of two players each (doubles). The game involves using a racket to hit a ball over a net into the opponent's court."), Document(metadata={'title': 'Basic Rules'}, page_content='- A match can be played as best of three or five sets.\n- Each set consists of games, and each game consists of points.\n- Points are scored as **0 (Love), 15, 30, 40**, and then **game**.\n- A player must win a game by at least **two points**.\n- The ball must land within the designated court boundaries.'), Document(metadata={'title': 'Scoring System'}, page_content='```plaintext\n0 points  -> Love\n1 point   -> 15\n2 points  -> 30\n3 points  -> 40\n4 points  -> Game (if leading by 2)\nDeuce     -> 40-40 (must win two consecutive points to win the game)\nAdvantage -> If a player wins a point at deuce, they gain the advantage\n```'

In [59]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [60]:
#Step_3 : Generate Embeddings

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_chunk(chunk):
  return embedding_model.encode([chunk], normalize_embeddings = True)

In [61]:
sample_embedding = embed_chunk(text_chunks[1])

In [62]:
print(sample_embedding)

[[ 4.31284010e-02  1.37310177e-02  4.09373827e-02 -6.01209216e-02
  -1.10041574e-01  3.76272351e-02  6.25852793e-02  5.84306121e-02
   7.23148212e-02  1.39388949e-01 -8.46664459e-02  3.00853178e-02
  -8.12309701e-03  1.30597679e-02  2.84464806e-02 -3.28884982e-02
   1.71879120e-02 -6.70572161e-04  3.37188207e-02  3.48397344e-02
   6.43046619e-03 -6.19975515e-02  2.90324111e-02 -1.02226608e-01
  -2.37306301e-02 -7.56799418e-04 -4.04314809e-02  6.89286143e-02
  -7.99834058e-02  3.47235836e-02 -2.76865009e-02  1.23020830e-02
  -3.25942971e-02  4.52941097e-02 -1.90717980e-01  3.52993910e-03
  -1.76258311e-02  4.93887775e-02 -2.13666018e-02  1.78316943e-02
   3.80032547e-02 -3.17196436e-02  1.73272677e-02  4.86741289e-02
   7.67677743e-03  1.05700135e-01 -2.81893108e-02  1.15144230e-01
   4.63571064e-02 -5.87211810e-02 -5.86445630e-02  1.76011696e-02
   4.08595726e-02 -5.73294936e-03  1.10638216e-01  2.60122144e-03
   4.54247817e-02  1.46811316e-02  2.28215549e-02 -2.84174513e-02
   5.38855

In [63]:
print(text_chunks[1])

Tennis is a popular sport played between two players (singles) or two teams of two players each (doubles). The game involves using a racket to hit a ball over a net into the opponent's court.


In [64]:
print(sample_embedding[0])

[ 4.31284010e-02  1.37310177e-02  4.09373827e-02 -6.01209216e-02
 -1.10041574e-01  3.76272351e-02  6.25852793e-02  5.84306121e-02
  7.23148212e-02  1.39388949e-01 -8.46664459e-02  3.00853178e-02
 -8.12309701e-03  1.30597679e-02  2.84464806e-02 -3.28884982e-02
  1.71879120e-02 -6.70572161e-04  3.37188207e-02  3.48397344e-02
  6.43046619e-03 -6.19975515e-02  2.90324111e-02 -1.02226608e-01
 -2.37306301e-02 -7.56799418e-04 -4.04314809e-02  6.89286143e-02
 -7.99834058e-02  3.47235836e-02 -2.76865009e-02  1.23020830e-02
 -3.25942971e-02  4.52941097e-02 -1.90717980e-01  3.52993910e-03
 -1.76258311e-02  4.93887775e-02 -2.13666018e-02  1.78316943e-02
  3.80032547e-02 -3.17196436e-02  1.73272677e-02  4.86741289e-02
  7.67677743e-03  1.05700135e-01 -2.81893108e-02  1.15144230e-01
  4.63571064e-02 -5.87211810e-02 -5.86445630e-02  1.76011696e-02
  4.08595726e-02 -5.73294936e-03  1.10638216e-01  2.60122144e-03
  4.54247817e-02  1.46811316e-02  2.28215549e-02 -2.84174513e-02
  5.38855083e-02 -4.91009

In [65]:
print(len(sample_embedding[0]))

384


In [66]:
pip install chromadb

In [67]:
#Step_04 : Strining Emabeddings in ChromaDB

vector_db = Chroma.from_texts(text_chunks, HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2"), persist_directory="/tmp/chroma_db")

In [68]:
vector_db._collection.get()

{'ids': ['1087b486-2527-419c-b141-cd43b489f260',
  '218f9556-ae0e-4cf3-8896-38585e596bd1',
  '19945304-6206-48a1-a351-b98a928a02e7',
  '6896cdeb-df38-4555-b61a-fc162f2dc9b4',
  'b22fadd1-be2d-4074-91d6-3839f3c83b48',
  '75b3bb56-4702-4997-8a29-6a8480ee9341',
  'fbd9f72c-9de1-440a-872e-c54cf8845244',
  '39ad54ea-658d-498c-84aa-228a1e621578',
  '3a947d60-0c76-4f9d-9d8c-b9fd6350f5da',
  'ddb304f6-2649-42ee-a9ff-a40b51a4884e',
  '7f327563-3c44-4e83-99a1-ad2e9ebe205b',
  'c9e1b340-9f31-41f4-99a7-e7dce7a5d742',
  'f9d60c0d-231e-4148-8cea-e4bcb40f6586',
  'ca88d854-f1bc-4c5a-8ef1-52be2e61b819',
  'c132030b-4551-45e5-81a2-4901668e68d8',
  '975e193e-1f29-4e5f-bc7c-df54dc054e65',
  '8059f7e4-7a21-4dd5-9c15-86f204b918f2',
  'a0bd5fe6-567c-444c-a4a1-31e169e2bf0c',
  '7c5cd732-7c29-4a74-9f1b-b73d40147668',
  '1b336b1b-a556-4586-a33f-8e5bc81a2e67',
  '42b59ab5-3dd1-4c59-ab47-6237435dfbad'],
 'embeddings': None,
 'documents': ['# Tennis',
  "Tennis is a popular sport played between two players (singl

In [69]:
vector_db._collection.get(include=["embeddings","documents"])

{'ids': ['1087b486-2527-419c-b141-cd43b489f260',
  '218f9556-ae0e-4cf3-8896-38585e596bd1',
  '19945304-6206-48a1-a351-b98a928a02e7',
  '6896cdeb-df38-4555-b61a-fc162f2dc9b4',
  'b22fadd1-be2d-4074-91d6-3839f3c83b48',
  '75b3bb56-4702-4997-8a29-6a8480ee9341',
  'fbd9f72c-9de1-440a-872e-c54cf8845244',
  '39ad54ea-658d-498c-84aa-228a1e621578',
  '3a947d60-0c76-4f9d-9d8c-b9fd6350f5da',
  'ddb304f6-2649-42ee-a9ff-a40b51a4884e',
  '7f327563-3c44-4e83-99a1-ad2e9ebe205b',
  'c9e1b340-9f31-41f4-99a7-e7dce7a5d742',
  'f9d60c0d-231e-4148-8cea-e4bcb40f6586',
  'ca88d854-f1bc-4c5a-8ef1-52be2e61b819',
  'c132030b-4551-45e5-81a2-4901668e68d8',
  '975e193e-1f29-4e5f-bc7c-df54dc054e65',
  '8059f7e4-7a21-4dd5-9c15-86f204b918f2',
  'a0bd5fe6-567c-444c-a4a1-31e169e2bf0c',
  '7c5cd732-7c29-4a74-9f1b-b73d40147668',
  '1b336b1b-a556-4586-a33f-8e5bc81a2e67',
  '42b59ab5-3dd1-4c59-ab47-6237435dfbad'],
 'embeddings': array([[ 0.0227568 ,  0.05737348,  0.06708645, ..., -0.09128203,
          0.03132669,  0.02229

In [70]:
#Step_05: Set up a LLM

pipe = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct")

Device set to use cpu


In [71]:
#Step_06: Retrieval and Generation

def retrieve_and_generate(query, threshold=1):
  """Retrieves relevant context from the vector database and generates an answer."""
  search_results = vector_db.similarity_search_with_score(query, k=1)
  print(search_results)

  if not search_results or search_results[0][1] > threshold:
    return "I don't know the answer to that question. There is no avilable context in the VectorDB"

    retrieved_context = search_results[0][0].page_content
    similarity_score = search_results[0][1]
    print(f"Retrieved Text: {retrieved_context}")
    print(f"Similarity Score: {similarity_score}")

    prompt = f"Answer the question using the given context\nContext: {retrieved_context}\nQuestion: {query}\nAnswer:"
    print(prompt)
    response = pipe(prompt, max_new_tokens = 100)
    return response[0]["generated_text"]

In [72]:
question = "what are famous tournaments?"
response = retrieve_and_generate(question)
print(response)

[(Document(metadata={}, page_content='- **Grand Slam Events**:\n- Australian Open\n- French Open\n- Wimbledon\n- US Open'), 0.9606984257698059)]
None


In [73]:
question = "what is cricket?"
response = retrieve_and_generate(question)
print(response)

[(Document(metadata={}, page_content="Tennis is a popular sport played between two players (singles) or two teams of two players each (doubles). The game involves using a racket to hit a ball over a net into the opponent's court."), 1.066832423210144)]
I don't know the answer to that question. There is no avilable context in the VectorDB


In [74]:
question = "what is tennis?"
response = retrieve_and_generate(question)
print(response)

[(Document(metadata={}, page_content="Tennis is a popular sport played between two players (singles) or two teams of two players each (doubles). The game involves using a racket to hit a ball over a net into the opponent's court."), 0.29123157262802124)]
None


In [75]:
question = "what is the scoring system in tennis?"
response = retrieve_and_generate(question)
print(response)

[(Document(metadata={}, page_content='- A match can be played as best of three or five sets.\n- Each set consists of games, and each game consists of points.\n- Points are scored as **0 (Love), 15, 30, 40**, and then **game**.\n- A player must win a game by at least **two points**.\n- The ball must land within the designated court boundaries.'), 0.7015942931175232)]
None
